# Recipe 03 — Custom Steps
> Write your own pipeline step to implement any transformation logic.

| | |
|---|---|
| **Module** | `anchor.pipeline` |
| **Key classes** | `PipelineStep`, `ContextPipeline` |
| **Difficulty** | Intermediate |

In [ ]:
from anchor.pipeline import ContextPipeline, PipelineStep
from anchor.models import QueryBundle, ContextItem, SourceType
from anchor.formatters import GenericTextFormatter

## 1 — Define a custom step function
A step function takes `(items, query, **kwargs)` and returns a filtered
or transformed list of `ContextItem` objects.

In [ ]:
def score_filter(items, query, **kwargs):
    """Keep only items with score above 0.3."""
    return [item for item in items if (item.score or 0) > 0.3]

print(f"Function name: {score_filter.__name__}")

## 2 — Wrap it in a `PipelineStep`
Give the step a name so diagnostics can track it.

In [ ]:
step = PipelineStep(name="score_filter", fn=score_filter)
print(f"Step: {step.name}")

## 3 — Create sample data
We generate items with varying scores to see the filter in action.

In [ ]:
items = [
    ContextItem(id="a", content="High quality result",
               source=SourceType.RETRIEVAL, score=0.9, token_count=8),
    ContextItem(id="b", content="Medium quality result",
               source=SourceType.RETRIEVAL, score=0.5, token_count=8),
    ContextItem(id="c", content="Low quality result",
               source=SourceType.RETRIEVAL, score=0.1, token_count=8),
]
print(f"Created {len(items)} test items")

## 4 — A content-dedup step
Custom steps can implement any logic. Here we deduplicate by content hash.

In [ ]:
import hashlib

def dedup_step(items, query, **kwargs):
    """Remove duplicate items based on content hash."""
    seen = set()
    unique = []
    for item in items:
        h = hashlib.md5(item.content.encode()).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(item)
    return unique

print("Dedup step defined")

## 5 — A priority-boost step
Boost priority for items whose content matches a keyword.

In [ ]:
def keyword_boost(items, query, **kwargs):
    """Boost priority when content contains the query string."""
    boosted = []
    for item in items:
        if query.query_str.lower() in item.content.lower():
            boosted.append(ContextItem(
                id=item.id, content=item.content,
                source=item.source, score=item.score,
                priority=(item.priority or 0) + 10,
                token_count=item.token_count,
            ))
        else:
            boosted.append(item)
    return boosted

print("Keyword boost step defined")

## 6 — Plug custom steps into a pipeline
Custom steps compose just like built-in ones.

In [ ]:
from anchor.pipeline.steps import retriever_step

# Mock retriever returning our sample items
class SampleRetriever:
    def retrieve(self, query, top_k=5):
        return [
            ContextItem(id=f"doc-{i}", content=f"Doc about {query.query_str} #{i}",
                        source=SourceType.RETRIEVAL, score=round(0.9 - i*0.2, 2),
                        token_count=12)
            for i in range(top_k)
        ]

pipeline = ContextPipeline(max_tokens=4096)
pipeline.add_system_prompt("You are a helpful assistant.")
pipeline.with_formatter(GenericTextFormatter())

pipeline.add_step(retriever_step("fetch", retriever=SampleRetriever(), top_k=5))
pipeline.add_step(PipelineStep(name="score_filter", fn=score_filter))
pipeline.add_step(PipelineStep(name="dedup", fn=dedup_step))
pipeline.add_step(PipelineStep(name="boost", fn=keyword_boost))

query = QueryBundle(query_str="Doc")
result = pipeline.build(query)

print(f"Items after all steps: {len(result.window.items)}")
for item in result.window.items:
    print(f"  {item.id}  score={item.score}  priority={item.priority}")

## Key Takeaways
- Any `(items, query, **kwargs) -> list[ContextItem]` function is a valid step.
- Wrap it with `PipelineStep(name=..., fn=...)` and call `pipeline.add_step()`.
- Custom steps compose seamlessly with built-in steps.

**Next:** [Async Pipelines](04_async_pipelines.ipynb)